## Creating a remote function to test RemoteFunctionStep

In [1]:
import mlrun

project = mlrun.get_or_create_project("test-notebooks", user_project=True)

> 2026-03-09 06:59:20,278 [info] Project loaded successfully: {"project_name":"test-notebooks-iguazio"}


In [2]:
%%writefile remote_function.py

def handler(context, event):
    print("Hello World !")
    print(f"event is : {event}")
    event.body['hey'] = "ho"
    print(f"event is : {event}")
    return event

Overwriting remote_function.py


In [3]:
fn = project.set_function(name='remote-function', kind="remote", image="python:3.11", func="remote_function.py")

In [4]:
address = project.deploy_function(fn)

> 2026-03-09 06:59:20,336 [info] Starting remote function deploy
2026-03-09 06:59:20  (info) Deploying function
2026-03-09 06:59:20  (info) Building
2026-03-09 06:59:20  (info) Staging files and preparing base images
2026-03-09 06:59:20  (warn) Using user provided base image, runtime interpreter version is provided by the base image
2026-03-09 06:59:20  (info) Building processor image
2026-03-09 07:00:03  (info) Build complete
2026-03-09 07:00:11  (info) Function deploy complete
> 2026-03-09 07:00:13,377 [info] Successfully deployed function: {"external_invocation_urls":["test-notebooks-iguazio-remote-function.default-tenant.app.vmdev67.lab.iguazeng.com/"],"internal_invocation_urls":["nuclio-test-notebooks-iguazio-remote-function.default-tenant.svc.cluster.local:8080"]}


In [5]:
fn.invoke("", body={"Iguazio": "MLRun"})

b'{"body": {"Iguazio": "MLRun", "hey": "ho"}, "content_type": "application/json", "trigger": {"kind": "http", "name": "http"}, "fields": {}, "headers": {"Content-Length": "20", "Content-Type": "application/json", "User-Agent": "python-requests/2.32.4", "Accept-Encoding": "gzip, deflate", "Accept": "*/*", "Connection": "keep-alive", "X-Nuclio-Target": "test-notebooks-iguazio-remote-function", "Host": "nuclio-test-notebooks-iguazio-remote-function.default-tenant.svc.cluster.local:8080"}, "id": "bba3fa4d-8259-47c7-99b3-7b1519bb77e4", "method": "POST", "path": "/", "size": 20, "timestamp": "2026-03-09 07:00:13", "url": "", "shard_id": -1, "num_shards": 0, "type": "", "type_version": "", "version": "", "last_in_batch": false, "offset": 0, "topic": ""}'

## Creating a serving graph with the new create function as a RemoteFunctionStep

In [6]:
from mlrun.serving.remote import RemoteFunctionStep


step = RemoteFunctionStep(fn=fn.metadata.name, project_name=project.name)

main_fn = mlrun.new_function(name="test-nuclio-remote-step", kind="serving")

flow = main_fn.set_topology("flow", engine="async", exist_ok=True)

flow.to(step).respond()

### Testing Locally

In [7]:
server = main_fn.to_mock_server()

try:
    resp = server.test(body={"Iguazio": "MLRun"})
except Exception as e:
    raise e
finally:
    server.wait_for_completion()
assert resp['body'] == {"Iguazio": "MLRun", 'hey': 'ho'}

> 2026-03-09 07:00:13,594 [warning] run command, file or code were not specified
> 2026-03-09 07:00:15,727 [info] Forwarding termination signal from step 'SyncEmitSource' to steps: RemoteFunctionStep
> 2026-03-09 07:00:15,728 [info] Sending termination signal to ConcurrentJobExecution worker belonging to step 'RemoteFunctionStep'
> 2026-03-09 07:00:15,728 [info] Awaiting termination of ConcurrentJobExecution worker belonging to step 'RemoteFunctionStep'
> 2026-03-09 07:00:15,729 [info] Terminating ConcurrentJobExecution worker belonging to step 'RemoteFunctionStep'
> 2026-03-09 07:00:15,730 [info] Terminated ConcurrentJobExecution worker belonging to step 'RemoteFunctionStep'
> 2026-03-09 07:00:15,731 [info] Forwarding termination signal from step 'RemoteFunctionStep' to steps: Complete


### Testing Remotely

In [8]:
main_address = main_fn.deploy()

> 2026-03-09 07:00:15,738 [info] Starting remote function deploy
2026-03-09 07:00:16  (info) Deploying function
2026-03-09 07:00:16  (info) Building
2026-03-09 07:00:16  (info) Staging files and preparing base images
2026-03-09 07:00:16  (warn) Using user provided base image, runtime interpreter version is provided by the base image
2026-03-09 07:00:16  (info) Building processor image
2026-03-09 07:01:35  (info) Build complete
2026-03-09 07:01:43  (info) Function deploy complete
> 2026-03-09 07:01:47,565 [info] Model endpoint creation task completed with state succeeded
> 2026-03-09 07:01:47,566 [info] Successfully deployed function: {"external_invocation_urls":["test-notebooks-iguazio-test-nuclio-remote-step.default-tenant.app.vmdev67.lab.iguazeng.com/"],"internal_invocation_urls":["nuclio-test-notebooks-iguazio-test-nuclio-remote-step.default-tenant.svc.cluster.local:8080"]}


In [9]:
resp = main_fn.invoke("/", body={"Iguazio": "MLRun"})

In [10]:
try:
    resp = main_fn.invoke("/", body={"Iguazio": "MLRun"})
except Exception as e:
    raise e
assert resp['body'] == {"Iguazio": "MLRun", 'hey': 'ho'}